In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold
import albumentations as A

# 1. Load and Format Data
print("Loading data...")
train_df = pd.read_csv('/kaggle/input/competitions/digit-recognizer/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/digit-recognizer/test.csv')

y_train = train_df['label'].values
# Convert to float32 for model stability
X_train = train_df.drop('label', axis=1).values.astype('float32')
X_test = test_df.values.astype('float32')

# Reshape to 28x28x1 and normalize to [0, 1]
X_train = X_train.reshape(-1, 28, 28, 1) / 255.0
X_test = X_test.reshape(-1, 28, 28, 1) / 255.0

# 2. Define Albumentations Pipeline
# We apply a slight shift, scale, and rotation. 
# Digits are shape-sensitive, so we avoid flips or extreme distortions.
transform = A.Compose([
    A.ShiftScaleRotate(
        shift_limit=0.1, 
        scale_limit=0.1, 
        rotate_limit=15, 
        border_mode=0, # Pad with constant value (black)
        value=0,
        p=0.75
    )
])

# 3. Custom Data Generator for Albumentations
class AlbumentationsGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, y, batch_size=64, augment=False):
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.augment = augment
        self.indices = np.arange(len(self.X))
        np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        X_batch = self.X[batch_indices]
        y_batch = self.y[batch_indices]

        if self.augment:
            X_batch_aug = []
            for img in X_batch:
                # Albumentations handles the (28, 28, 1) NumPy arrays seamlessly
                augmented = transform(image=img)
                X_batch_aug.append(augmented['image'])
            X_batch = np.array(X_batch_aug)

        return X_batch, y_batch

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

# 4. Model Creation Function
# Encapsulated in a function so we can generate a fresh model for each fold
def build_model():
    model = models.Sequential([
        layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.BatchNormalization(), # Added Batch Normalization for faster, stabler training
        layers.MaxPooling2D(pool_size=(2, 2)),
        
        layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),
        
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# 5. K-Fold Cross Validation & Ensembling Setup
n_splits = 5
epochs_per_fold = 15
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Array to store the combined predictions for the test set
ensemble_predictions = np.zeros((X_test.shape[0], 10))

# 6. The Training Loop
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n========== FOLD {fold + 1}/{n_splits} ==========")
    
    # Split data for this fold
    X_tr, y_tr = X_train[train_idx], y_train[train_idx]
    X_val, y_val = X_train[val_idx], y_train[val_idx]
    
    # Initialize Data Generators
    train_gen = AlbumentationsGenerator(X_tr, y_tr, batch_size=64, augment=True)
    val_gen = AlbumentationsGenerator(X_val, y_val, batch_size=64, augment=False)
    
    # Build a fresh model
    model = build_model()
    
    # Add a Learning Rate Scheduler for fine-tuning performance
    lr_reduction = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', 
        patience=3, 
        verbose=1, 
        factor=0.5, 
        min_lr=0.00001
    )
    
    # Train
    model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs_per_fold,
        callbacks=[lr_reduction],
        verbose=1
    )
    
    # Generate predictions for the test set from THIS fold's model
    print(f"Adding Fold {fold + 1} predictions to ensemble...")
    fold_preds = model.predict(X_test)
    
    # Average the probabilities (Soft Voting)
    ensemble_predictions += fold_preds / n_splits

# 7. Final Submission
print("\nCreating final ensemble submission...")
# Pick the highest averaged probability class for each image
final_results = np.argmax(ensemble_predictions, axis=1)

submission = pd.DataFrame({
    'ImageId': range(1, len(final_results) + 1),
    'Label': final_results
})

submission.to_csv('submission_kfold_albumentations.csv', index=False)
print("Done! Saved as 'submission_kfold_albumentations.csv'.")

2026-04-26 05:59:27.258139: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777183167.451710      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777183167.507971      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777183167.955680      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777183167.955716      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777183167.955719      23 computation_placer.cc:177] computation placer alr

Loading data...


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_23/723812188.py:26: UserWarning: Argument(s) 'value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1777183201.897991      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0



========== FOLD 1/5 ==========
Epoch 1/15


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1777183205.498782      67 service.cc:152] XLA service 0x79a878004c90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777183205.498830      67 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1777183205.912241      67 cuda_dnn.cc:529] Loaded cuDNN version 91002


 14/525 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.3684 - loss: 2.4367

I0000 00:00:1777183208.753100      67 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


525/525 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.8194 - loss: 0.5928 - val_accuracy: 0.9650 - val_loss: 0.1136 - learning_rate: 0.0010
Epoch 2/15
525/525 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.9539 - loss: 0.1546 - val_accuracy: 0.9787 - val_loss: 0.0670 - learning_rate: 0.0010
Epoch 3/15
525/525 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9652 - loss: 0.1107 - val_accuracy: 0.9769 - val_loss: 0.0784 - learning_rate: 0.0010
Epoch 4/15
525/525 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9708 - loss: 0.1009 - val_accuracy: 0.9867 - val_loss: 0.0457 - learning_rate: 0.0010
Epoch 5/15
525/525 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.9752 - loss: 0.0793 - val_accuracy: 0.9912 - val_loss: 0.0393 - learning_rate: 0.0010
Epoch 6/15
525/525 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.9763 - loss: 0.0803 - val_accuracy: 0.9877 - val_loss: 0.0475 - learning_rate: 0.0010
Epoch 7/15
525/525 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.9785 - loss: 0.0703 - val